In [ ]:
%matplotlib inline
import torch
from conformal.specbridge import mass_spec_gym_candidates, mass_spec_gym_dataset

# 1. Precompute Model Predictions

**This takes a long time ! Run it once, and then reuse the saved CSV predictions**

Load datasets

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

dataset = mass_spec_gym_dataset()
candidates = mass_spec_gym_candidates()

Load model

In [ ]:
from argparse import Namespace
from specbridge.adapters.dreams_adapter import load_dreams_encoder
from specbridge.models.mapper import DreamsToMolCondition
from conformal.specbridge import SmilesPredictor


dreams_encoder = load_dreams_encoder(
    "Models/SpecBridge/runs/DreaMS/ssl_model.ckpt", d_in=2048, d_out=1024
)

dtmc = DreamsToMolCondition(
    dreams_encoder,
    d_out=2048,
    mapper_hidden=2048,
    gaussian=False,
    mol_space="chemberta",
    chemberta_model="Derify/ChemBERTa_augmented_pubchem_13m",
    args=Namespace(n_blocks=8),
).eval()

dtmc.load_state_dict(
    torch.load(
        "Models/SpecBridge/runs/msgym/SpecBridge_MSGYM_checkpoint.pt",
        map_location="cpu",
        weights_only=False,
    )["model"],
    strict=False,
)

for p in dtmc.parameters():
    p.requires_grad = False


model = SmilesPredictor(dtmc, device)

Smiles prediction example

In [ ]:
spectrum = dataset[10]
ground_truth = spectrum["smiles"]

print(f"Sample: {spectrum['title']}")
print(f"True SMILES: {ground_truth}")
print(f"Number of candidates: {len(candidates[ground_truth])}")

predictions = model.forward(spectrum, candidates[ground_truth], top_k=5)


print("\nTop 5 Predictions:")
for i, (smiles, score) in enumerate(predictions, 1):
    is_correct = "✓" if smiles == ground_truth else "✗"
    print(f"{i}. {is_correct} {smiles[:60]} | score: {score:.4f}")

Run top-1 predictions on the test set

In [ ]:
import csv
from tqdm import tqdm

with open("predictions.csv", "w", newline="") as csvfile:
    writer = csv.writer(csvfile)
    writer.writerow(["Ground Truth", "Prediction"])

    for sample in tqdm(iter(dataset), total=len(dataset)):
        sample_candidates = candidates[sample["smiles"]]

        (prediction, _), *_ = model.forward(sample, sample_candidates)
        writer.writerow([sample["smiles"], prediction])

# 2. Calibration

### Standard Conformal Predictions

In [ ]:
from typing import Iterable

import numpy as np
import polars as pl

from conformal.calibrate import ConformalPredictor
from conformal.fgw import FGW
from conformal.graph import Graph
from conformal.specbridge import mass_spec_gym_candidates

fgw = FGW(cost="laplacian")
predictor = ConformalPredictor(fgw)

df = pl.read_csv("predictions.csv")

candidates = mass_spec_gym_candidates()
truths = df["Ground Truth"].to_numpy()
preds = df["Prediction"].to_numpy()

Fit the calibration model

In [ ]:
calibration_rate = 0.6
rng = np.random.default_rng(seed=42)
mask = rng.random(len(df)) < calibration_rate

calib_truths = truths[mask]
calib_preds = preds[mask]

# fit the predictor
predictor.fit(
    data=Graph.stream_from_smiles_pair(calib_preds, calib_truths),
    n=len(calib_truths),
)

print(f"Predictor threshold: {predictor.threshold:.3f}")

Evaluate the model

In [ ]:
valid_truths = truths[~mask]
valid_preds = preds[~mask]


def stream_candidates() -> Iterable[Iterable[Graph]]:
    for t in valid_truths:
        yield Graph.stream_from_smiles(candidates[t])


# compute metrics
metrics = predictor.predict(
    data=Graph.stream_from_smiles_pair_and_candidates(
        valid_preds, valid_truths, [candidates[t] for t in valid_truths]
    ),
    n=len(valid_truths),
)

# save to disk. See conformal/plot.py to reproduce the draft plots with matplotlib
metrics.save("metrics-laplacian.pkl")

print(f"Coverage: {metrics.coverage:.3f}")
print(f"Mean set size: {metrics.mean_set_size:.3f}")
print(f"Median set size: {metrics.median_set_size:.3f}")
print(f"Mean reduction: {metrics.mean_reduction:.3f}")
print(f"Median reduction: {metrics.median_reduction:.3f}")
print(f"Empty set rate: {metrics.empty_rate:.3f}")
